In [ ]:
import cv2
import numpy as np
import pprint
from pathlib import Path
from matplotlib import pyplot as plt
from cap_augmentation import CAP_AUG, CAP_AUG_Multiclass
from cap_augmentation.utils import show_image, draw_bboxes

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'pyproject.toml').exists():
    for parent in REPO_ROOT.parents:
        if (parent / 'pyproject.toml').exists():
            REPO_ROOT = parent
            break


# Load data

## Load destination image

In [ ]:
DEST_DATASET_ROOT = Path('/dataset/kitti_format/vbd/png_keep_ratio/train/')

In [ ]:
dest_image_name = '7ffe83778375bb8229b12c2ad4570c0e.png'

dest_image_name = DEST_DATASET_ROOT / dest_image_name
image = cv2.imread(str(dest_image_name))

In [ ]:
show_image(image)

## Get list of source images

In [ ]:
class_idx = 0

In [ ]:
CLASS_DATASET_ROOT = REPO_ROOT / f'data/vinbig_dataset/{class_idx}'
SOURCE_IMAGES = sorted(list(CLASS_DATASET_ROOT.glob('*.png')))

In [ ]:
prob_map_path = str(REPO_ROOT / f'data/vinbig_dataset/analytics/{class_idx}.npy')
prob_map = np.load(prob_map_path,allow_pickle='TRUE').item()

## Example in pixel coordinates

Note : If bev_transform parameter is None, then x_range and y_range must be a list of integers and these parameters will mean range in x and y axis in the image coordinate system. h_range parameter also must be a list of interegers (pixel sizes of images)

In [ ]:
cap_aug_pixels = CAP_AUG(SOURCE_IMAGES,  bev_transform=None,
                                         probability_map=prob_map['probability_map'],
                                         mean_h_norm=prob_map['mean_h'],
                                         histogram_matching=True,
                                         n_objects_range=[5,10], 
                                         blending_coeff=0.5)

In [ ]:
result_image, result_coords, semantic_mask, instance_mask = cap_aug_pixels(image)
show_image(result_image)

In [ ]:
result_coords

### Draw bounding boxes

In [ ]:
result_image_vis, result_mask = draw_bboxes(result_image, result_coords)
show_image(result_image_vis)

# Test multiclass

In [ ]:
import cv2
import numpy as np
import pprint
from pathlib import Path
from matplotlib import pyplot as plt
from cap_augmentation import CAP_AUG, CAP_AUG_Multiclass
from cap_augmentation.utils import show_image, draw_bboxes

In [ ]:
DEST_DATASET_ROOT = Path('/dataset/kitti_format/vbd/png_keep_ratio/train/')
dest_image_name = '7ffe83778375bb8229b12c2ad4570c0e.png'

dest_image_name = DEST_DATASET_ROOT / dest_image_name
image = cv2.imread(str(dest_image_name))

In [ ]:
class_idxs = [0,3,11]
probabilities = [0.9,0.9,0.9]
cap_augs = []

for class_idx in class_idxs:
    CLASS_DATASET_ROOT = REPO_ROOT / f'data/vinbig_dataset/{class_idx}'
    SOURCE_IMAGES = sorted(list(CLASS_DATASET_ROOT.glob('*.png')))

    prob_map_path = str(REPO_ROOT / f'data/vinbig_dataset/analytics/{class_idx}.npy')
    prob_map = np.load(prob_map_path,allow_pickle='TRUE').item()
    
    cap_aug_pixels = CAP_AUG(SOURCE_IMAGES,  bev_transform=None,
                                         probability_map=prob_map['probability_map'],
                                         mean_h_norm=prob_map['mean_h'],
                                         histogram_matching=True,
                                         random_h_flip=True,
                                         random_v_flip=True,
                                         n_objects_range=[1,3], 
                                         blending_coeff=0.6)
    cap_augs.append(cap_aug_pixels)

In [ ]:
cap_aug_multiclass = CAP_AUG_Multiclass(cap_augs, probabilities, class_idxs)

In [ ]:
result_image, result_coords, semantic_mask, instance_mask = cap_aug_multiclass(image)
show_image(result_image)

In [ ]:
result_coords

In [ ]:
result_image_vis, result_mask = draw_bboxes(result_image, result_coords)
show_image(result_image_vis)